# CRISP-DM — Predicción de Asistencia a Eventos (CertyPro)

Notebook que recorre las **6 fases de CRISP-DM** aplicadas a la predicción de
asistencia de participantes a eventos de capacitación.

- **Sistema libre:** Python, pandas, scikit-learn, Matplotlib (todas libres).
- **Consumo de API:** Nager.Date (feriados de Perú), pública y sin API key.
- **Reproducible:** semilla fija (RANDOM_SEED = 42).

> Informe metodológico completo en `INFORME_CRISP_DM.md`.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import pandas as pd, matplotlib.pyplot as plt
from crispdm.config import RANDOM_SEED, FEATURE_COLUMNS, TARGET
from crispdm.data_source import load_raw
from crispdm.features import build_feature_matrix
from crispdm.model import train_and_evaluate, load_model
print('Semilla de reproducibilidad:', RANDOM_SEED)


## Fase 1 — Comprensión del negocio

**Problema:** inscritos que no asisten (ausentismo) → desperdicio de aforo y recursos.

**Objetivo de minería:** clasificar cada inscripción como *asistirá / no asistirá*.

**Criterio de éxito:** ROC-AUC ≥ 0.75 en validación cruzada.


## Fase 2 — Comprensión de los datos

Unidad de análisis = inscripción (participante en un evento). Variable objetivo `attended`.


In [ ]:
raw, source = load_raw(prefer_orm=False)  # False = usa datos sintéticos reproducibles
print('Origen:', source, '| Filas:', len(raw))
print('Tasa de asistencia global: {:.1%}'.format(raw[TARGET].mean()))
raw.head()


In [ ]:
raw[TARGET].value_counts().plot(kind='bar', color=['#ef4444','#22c55e'])
plt.title('Distribución de la variable objetivo'); plt.xlabel('0=No asistió  1=Asistió'); plt.show()


## Fase 3 — Preparación de los datos

Ingeniería de variables + **consumo de la API de feriados** (`near_holiday`).
Se evita *data leakage*: el historial del participante usa solo eventos previos.


In [ ]:
X, y, enriched = build_feature_matrix(raw, use_network=True)  # consulta la API de feriados
print('Matriz:', X.shape)
print('Eventos cerca de feriado: {:.1%}'.format(enriched['near_holiday'].mean()))
X.head()


In [ ]:
enriched.groupby('category')[TARGET].mean().sort_values().plot(kind='barh', color='#3b82f6')
plt.title('Tasa de asistencia por categoría'); plt.xlabel('Tasa'); plt.show()


## Fase 4 — Modelado

Regresión Logística vs. Random Forest, con validación cruzada de 5 folds.


In [ ]:
report = train_and_evaluate(X, y)
pd.DataFrame(report['results']).T[['roc_auc','f1','accuracy','cv_roc_auc_mean']]


## Fase 5 — Evaluación


In [ ]:
best = report['best_model_name']
print('Modelo ganador:', best)
m = report['results'][best]
print('ROC-AUC: {roc_auc} | F1: {f1} | CV-AUC: {cv_roc_auc_mean}'.format(**m))
meta = 'ALCANZADA' if m['cv_roc_auc_mean']>=0.75 else 'NO alcanzada'
print('Meta (CV-AUC>=0.75):', meta)
pd.DataFrame(report['feature_importances']).head(8)


In [ ]:
imp = pd.DataFrame(report['feature_importances']).head(10)[::-1]
plt.barh(imp['feature'], imp['importance'], color='#8b5cf6')
plt.title('Importancia de variables'); plt.tight_layout(); plt.show()


## Fase 6 — Despliegue

El modelo se guarda en `models/attendance_model.joblib` y se expone vía la API Django:
`POST /api/analytics/predict-attendance/`. Aquí lo probamos directamente.


In [ ]:
modelo = load_model()
buen_caso = {'duration_hours':8,'max_capacity':50,'lead_time_days':20,'participant_prior_events':3,
             'participant_prior_rate':0.8,'event_dow':2,'event_month':6,'is_public':1,
             'near_holiday':0,'was_invited':1,'invitation_accepted':1,'category':'Tecnologia'}
mal_caso = dict(buen_caso, lead_time_days=1, duration_hours=40, near_holiday=1, invitation_accepted=0, is_public=0)
for nombre, caso in [('Buen caso', buen_caso), ('Mal caso', mal_caso)]:
    Xc = pd.DataFrame([caso], columns=FEATURE_COLUMNS)
    print(nombre, '-> prob. asistencia =', round(float(modelo.predict_proba(Xc)[0,1]),3))


---
**Conclusión.** El pipeline cumple los 4 requisitos: sistema libre, consumo de API,
metodología CRISP-DM completa y total replicabilidad (semilla fija + un comando).
